# HackerNews Analytics Pro v4.0

NLP, topic discovery, chronological score modelling and reproducible report export. Demo records are synthetic.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT).exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from hn_analytics import load_stories, clean_stories, sentiment_analysis, discover_topics, train_score_model, export_report
raw=load_stories(ROOT/"demo_stories.jsonl")
df=clean_stories(raw)
print(f"Loaded {len(df):,} records")


## 1. Lexicon sentiment

Transparent keyword baseline for title text. It is not a pretrained language model and may miss sarcasm/context.


In [ ]:
sentiment_df=sentiment_analysis(df)
sentiment_df["sentiment_label"].value_counts()


## 2. Unsupervised topic discovery

TF-IDF + NMF extracts recurring title terms. Topic labels are keyword summaries, not human-verified categories.


In [ ]:
topic_df, topic_summary=discover_topics(df, n_topics=5)
display(topic_summary)
display(topic_df[["title","topic_id","topic_terms"]].head(10))


## 3. Score modelling

Random Forest baseline with chronological 80/20 holdout. Uses title, domain, type and UTC hour only; score/comments are excluded from predictors. The target is log1p(score).


In [ ]:
try:
    model, metrics=train_score_model(df)
    metrics
except ValueError as exc:
    print(f"Not enough scored rows for model evaluation: {exc}")


## 4. Export executive report


In [ ]:
summary=export_report(df, ROOT/"reports")
summary


## 5. Interpretation safeguards

- Bundled data is synthetic; do not report its outputs as real platform findings.
- Sentiment is a simple lexicon heuristic, not a psychological or editorial judgement.
- Topic modelling is exploratory and depends on sample size and vocabulary.
- Predictive metrics are only meaningful on representative, temporally held-out real data.
- Correlation and prediction do not establish causality.
